In [ ]:
import cv2
import numpy as np

def estimate_atmospheric_light(image, method='max'):
    """
    Estimates the atmospheric light (A) in an image, which is a crucial parameter
    for haze removal.

    Args:
        image (numpy.ndarray): The input image (BGR format).
        method (str, optional):  Method to estimate A.
            'max':  Selects the brightest pixels.  Good for dense haze.
            'mean': Averages brightest pixels.  Smoother result.
            Defaults to 'max'.

    Returns:
        numpy.ndarray: A 1x3 array representing the estimated atmospheric light (A).
    """
    height, width = image.shape[:2]
    pixels = image.reshape(height * width, 3)  # Reshape to list of pixels

    # Select top 0.1% brightest pixels
    num_brightest = int(height * width * 0.001)
    brightest_indices = np.argpartition(np.sum(pixels, axis=1), -num_brightest)[-num_brightest:]
    brightest_pixels = pixels[brightest_indices]

    if method == 'max':
        # A = brightest pixel
        A = np.max(brightest_pixels, axis=0)
    elif method == 'mean':
        # A = mean of brightest pixels
        A = np.mean(brightest_pixels, axis=0)
    else:
        raise ValueError(f"Invalid method: {method}.  Choose 'max' or 'mean'.")
    return A

def estimate_dark_channel(image, patch_size=15):
    """
    Estimates the dark channel of an image.  The dark channel is a grayscale
    image where each pixel's intensity represents the minimum intensity
    in a local patch surrounding that pixel.  It's used to estimate haze thickness.

    Args:
        image (numpy.ndarray): The input image (BGR format).
        patch_size (int, optional): Size of the local patch.  Larger patch_size
            gives thicker haze estimation. Defaults to 15.

    Returns:
        numpy.ndarray: The dark channel image (grayscale).
    """
    height, width = image.shape[:2]
    dark_channel = np.zeros((height, width), dtype=np.uint8)

    # Pad the image to handle boundary pixels
    pad_size = patch_size // 2
    padded_image = np.pad(image, ((pad_size, pad_size), (pad_size, pad_size), (0, 0)), 'edge')

    for y in range(height):
        for x in range(width):
            # Extract the local patch
            patch = padded_image[y:y + patch_size, x:x + patch_size]
            # Get the minimum value across the color channels for each pixel in the patch
            min_values = np.min(patch, axis=2)
            # Find the minimum value within the patch
            dark_channel[y, x] = np.min(min_values)
    return dark_channel

def dehaze(image, A, t0=0.1, patch_size=15):
    """
    Removes haze from an image using the Dark Channel Prior.

    Args:
        image (numpy.ndarray): The input image (BGR format).
        A (numpy.ndarray): The atmospheric light (1x3 array).
        t0 (float, optional): Minimum transmission.  Helps prevent division by zero.
            Defaults to 0.1.
        patch_size (int, optional): Size of the local patch for dark channel
            estimation. Defaults to 15.

    Returns:
        numpy.ndarray: The dehazed image (BGR format).
    """
    height, width = image.shape[:2]
    dark_channel = estimate_dark_channel(image, patch_size)

    # Estimate transmission (t)
    t = 1 - 0.95 * (dark_channel / np.max(A))  # 0.95 is a common parameter
    t = np.clip(t, t0, 1)  # Ensure transmission is within [t0, 1]

    # Dehaze the image
    dehazed_image = np.zeros_like(image, dtype=np.uint8)
    for channel in range(3):
        dehazed_image[:, :, channel] = (
            (image[:, :, channel].astype(np.float32) - A[channel]) / t + A[channel]
        ).astype(np.uint8)

    return dehazed_image

def remove_fog_from_video(video_path, output_path="dehazed_video.mp4", method='max'):
    """
    Removes fog from a video and saves the processed video.

    Args:
        video_path (str): Path to the input video file.
        output_path (str, optional): Path to save the dehazed video.
            Defaults to "dehazed_video.mp4".
        method (str, optional): Method to estimate atmospheric light.
            'max':  Selects the brightest pixels.
            'mean': Averages brightest pixels.
            Defaults to 'max'.
    """
    # Open the video file
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"Error: Could not open video file at {video_path}")
        return

    # Get video properties (width, height, FPS)
    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)

    # Define the codec and create a VideoWriter object to save the output
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')  # Use 'mp4v' for .mp4, 'XVID' for .avi
    out = cv2.VideoWriter(output_path, fourcc, fps, (frame_width, frame_height))
    if not out.isOpened():
        print(f"Error: Could not create output video file at {output_path}")
        cap.release()
        return

    frame_count = 0
    while True:
        # Read the next frame from the video
        ret, frame = cap.read()
        if not ret:
            # End of video or error
            break
        frame_count += 1

        # Dehaze the frame
        try:
            A = estimate_atmospheric_light(frame, method)
            dehazed_frame = dehaze(frame, A)
        except Exception as e:
            print(f"Error processing frame {frame_count}: {e}")
            # Consider handling the error: skip frame, use previous frame, etc.
            continue  # Skip to the next frame

        # Write the dehazed frame to the output video
        out.write(dehazed_frame)

        # Optionally display the original and dehazed frames (for debugging or preview)
        cv2.imshow("Original Frame", frame)
        cv2.imshow("Dehazed Frame", dehazed_frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):  # Press 'q' to quit
            break

    # Release video capture and writer
    cap.release()
    out.release()
    cv2.destroyAllWindows()
    print(f"Processed video saved to {output_path}")

if __name__ == "__main__":
    # Example usage:
    input_video = "foggy_video.mp4"  # Replace with your foggy video
    output_video = "dehazed_video.mp4"
    # Create a dummy video if the file does not exist
    dummy_video = np.zeros((100, 100, 3, 10), dtype=np.uint8) # 10 frames, 100x100, 3 channels
    for i in range(10):
        dummy_video[:,:,:,i] = (i+1)*25

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(input_video, fourcc, 10, (100, 100))
    for i in range(10):
        out.write(dummy_video[:,:,:,i])
    out.release()

    remove_fog_from_video(input_video, output_video, method='mean')